# 1.4 Host C++ 构建流程

## 本节学习目标

- 使用鲲鹏毕昇 clang++ 构建 C++17 程序
- 区分 Host 编译与 AI Core 编译

## 必要背景

OpenMP 和 MPI 项目首先是 Host C++ 工程。本实验明确使用鲲鹏毕昇 `clang++` 完成最小 Host 构建；MPI 工程仍通过 `mpicxx` wrapper 调用其配置的 Host 编译器。

## 构建边界

`clang++` 生成 Host C++ 可执行文件，`mpicxx` 负责 MPI Host 工程；`bisheng` 用于 Ascend C/AI Core 源。ACL 应用由 Host 编译器编译并链接 CANN 库。

## 编译最小 Host 程序

从当前章节目录执行下面的 Cell，并对照随后给出的检查点阅读输出。

In [ ]:
from pathlib import Path
import shutil
import subprocess

cxx = shutil.which("clang++")
if not cxx:
    raise RuntimeError("未找到鲲鹏毕昇 Host 编译器 clang++，请先完成 1.3 的安装步骤。")
version = subprocess.run([cxx, "--version"], check=True, capture_output=True, text=True).stdout
if "bisheng" not in version.lower():
    raise RuntimeError("当前 clang++ 不是鲲鹏毕昇 Host 编译器，请检查 PATH 顺序。")

Path("build").mkdir(exist_ok=True)
subprocess.run([cxx, "-std=c++17", "-O2", "src/hello_host.cpp", "-o", "build/hello_host"], check=True)
subprocess.run(["./build/hello_host"], check=True)


## 预期现象与结果分析

程序应输出 `host toolchain ready`。这个任务不需要 NPU，可以用于区分主机编译问题和 CANN 运行问题。

## 课后实践

分别写出普通 C++、OpenMP、MPI wrapper、ACL 应用和 Ascend C Kernel 的编译入口。

参考答案见 `answer/01.04_answer.md`。

In [ ]:
from pathlib import Path
print(Path('answer/01.04_answer.md').read_text(encoding='utf-8'))

## 实验步骤：把本节落实到 `src/`

1. 从 Notebook 当前目录确认 `src/` 存在。
2. 打开本节 Code Cell 定位的片段，在完整文件中找到所属函数和调用者。
3. 对照工程职责：`hello_host.cpp` 是 Host C++ 最小程序；`hello_world.asc` 用于识别毕昇/Ascend C 编译边界。
4. 执行到本节对应阶段：检查环境与工具版本 → 阅读两个源码 → 编译运行 Host 程序 → 记录工具链 → 判断哪些步骤需要 CANN。
5. 每次只改变一个变量，固定输入、构建类型、warmup/repeat。
6. 记录：工具、版本/路径、退出码、Host 输出、Ascend C 编译条件。
7. 先检查退出码和正确性，再比较时间；历史结果不是本机输出。

### 本节完成标准

能够用真实文件和函数解释机制，给出可复现命令、至少一组结果或环境受限诊断，并说明结果如何进入下一节。
